<a href="https://colab.research.google.com/github/eunseojeon/AI_Coding_for_Autonomous_Driving_Class/blob/main/0731_%EC%95%99%EC%83%81%EB%B8%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import shutil
import os

folder = "/content"
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # 파일 또는 심볼릭 링크 삭제
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)  # 폴더(디렉토리) 삭제
    except Exception as e:
        print(f'Failed to delete {file_path}. Reason: {e}')

#위 코드는 아래 코드들에서 오류났을 때, 새로운 파일이 생기니까 그 파일을 다 삭제해주는 코드! (리셋느낌)

In [1]:
!pip install ensemble-boxes ultralytics opencv-python-headless==4.7.0.72 tqdm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 108.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curr

In [2]:
from google.colab import files

print("분석할 영상(mp4) 파일을 선택해서 업로드하세요.")
uploaded = files.upload()
video_path = [k for k in uploaded.keys() if k.endswith('.mp4')][0]


분석할 영상(mp4) 파일을 선택해서 업로드하세요.


Saving 교수님도로영상.mp4 to 교수님도로영상.mp4


In [4]:
import urllib.request
import os
yolo_weight = "yolov8n.pt"
if not os.path.exists(yolo_weight):
    url = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n.pt"
    print("YOLO 가중치 다운로드 중...")
    urllib.request.urlretrieve(url, yolo_weight)
    print("YOLOv8n 가중치 다운로드 완료.")

YOLO 가중치 다운로드 중...
YOLOv8n 가중치 다운로드 완료.


In [8]:
!pip uninstall numpy -y
!pip install numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 108.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albucore 0.0.24 requires opencv-python-headless>=4.9.0.80, but you have opencv-python-headless 4.7.0.72 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
albumentations 2.0.8 requires opencv-python-headless>=4.9.0.80, but you have opencv-python-headless 4.7.0.72 which is incompatible.


In [7]:
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
import cv2
import numpy as np
import os

yolo_weight = "yolov8n.pt" # Define yolo_weight here
yolo_model = YOLO(yolo_weight)
class_names = yolo_model.names
class_to_id = {name: i for i, name in enumerate(class_names)}
id_to_class = {v: k for k, v in class_to_id.items()}

In [10]:
from google.colab import files
uploaded = files.upload()
video_path = [k for k in uploaded.keys() if k.endswith('.mp4')][0]


Saving 교수님도로영상.mp4 to 교수님도로영상.mp4


In [14]:
class_names = yolo_model.names  # YOLOv8/YOLOv11 등 ultralytics라면 이렇게!
class_to_id = {name: i for i, name in enumerate(class_names)}
id_to_class = {v: k for k, v in class_to_id.items()}


In [38]:
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
import cv2
import numpy as np

# YOLOv8 모델 로드
yolo_model = YOLO(yolo_weight)
# ② 여기에 바로 아래 코드를 넣으세요!
if isinstance(yolo_model.names, dict):
    class_names = list(yolo_model.names.values())
else:
    class_names = yolo_model.names

class_to_id = {name: i for i, name in enumerate(class_names)}
id_to_class = {v: k for k, v in class_to_id.items()}

# 객체별 박스색/이름 설정
COLOR_TABLE = {
    'person': (255, 0, 0),        # 파랑
    'car': (0, 255, 0),           # 초록
    'truck': (0, 255, 255),       # 노랑
    'bus': (255, 255, 0),         # 하늘
    'motorcycle': (255, 0, 255),  # 보라
    'bicycle': (0, 128, 255),     # 주황
    'traffic light': (0, 0, 255), # 빨강
}
DEFAULT_COLOR = (200, 200, 200)   # 기타(회색)

# ==== (2) 여기! PeopleNet/TrafficNet 함수 정의 ====
def real_peoplenet_inference(frame):
    # 본인 모델 추론 코드
    return []

def real_trafficnet_inference(frame):
    # 본인 모델 추론 코드
    return []

In [39]:
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('ensemble.mp4', fourcc, fps, (w, h))

from tqdm import trange
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 100

def format_for_wbf(results, img_shape):
    boxes, scores, labels = [], [], []
    for c, b, s in results:
       if c not in class_to_id:
            print(f"[경고] '{c}'는 class_to_id에 없습니다! (YOLO class_names와 맞춰주세요!)")
       x_min = b[0]/img_shape[1]; y_min = b[1]/img_shape[0]
       x_max = b[2]/img_shape[1]; y_max = b[3]/img_shape[0]
       boxes.append([x_min, y_min, x_max, y_max])
       scores.append(float(s))
       labels.append(class_to_id.get(c, 0))
    return boxes, scores, labels

frame_idx = 0
for _ in trange(frame_count):
    ret, frame = cap.read()
    if not ret:
        break

     # 실제 모델 추론 결과를 PeopleNet, TrafficNet 결과 변수에 할당
    peoplenet_results = [
        ('person', [123, 50, 180, 210], 0.98),
        ('person', [210, 100, 265, 250], 0.91)
    ]
    trafficnet_results = [
        ('car', [310, 190, 420, 295], 0.93),
        ('truck', [30, 220, 80, 285], 0.77),
        ('traffic light', [450, 80, 470, 120], 0.81)
    ]

    # YOLO 추론 (BGR→RGB 변환)
    img_np = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    yolo_result = yolo_model(img_np)[0]
    yolo_preds = []
    for b in yolo_result.boxes:
        xyxy = b.xyxy[0].tolist()
        label = class_names[int(b.cls[0])]
        conf = float(b.conf[0])
        yolo_preds.append((label, xyxy, conf))

    # PeopleNet/TrafficNet(예시, 실제 사용시 각 모델 추론 결과로 교체)
    #peoplenet_results = [('person', [60+frame_idx, 40, 120+frame_idx, 180], 0.88)]
    #trafficnet_results = [('car', [65+frame_idx, 45, 130+frame_idx, 185], 0.75)]

    sources, scores, labels = [], [], []
    for results in [peoplenet_results, trafficnet_results, yolo_preds]:
        b, s, l = format_for_wbf(results, frame.shape)
        sources.append(b)
        scores.append(s)
        labels.append(l)

    if all([len(b)==0 for b in sources]):
        out.write(frame)
        frame_idx += 1
        continue

    # WBF 앙상블
    boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        sources, scores, labels, iou_thr=0.5, skip_box_thr=0.3
    )

    # [이 부분이 핵심!] 클래스별 이름/색상 자동 적용 박스그리기
    for box, score, lab in zip(boxes, fused_scores, fused_labels):
        lab_int = int(round(lab))
        class_name = id_to_class.get(lab_int, f"class{lab_int}")
        x1, y1, x2, y2 = int(box[0]*w), int(box[1]*h), int(box[2]*w), int(box[3]*h)
        color = COLOR_TABLE.get(class_name, DEFAULT_COLOR)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            frame, f"{class_name}:{score:.2f}", (x1, max(y1-10,0)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2
        )
    out.write(frame)
    frame_idx += 1

cap.release()
out.release()
print('ensemble.mp4 저장 완료')

  0%|          | 0/1174 [00:00<?, ?it/s]


0: 288x640 9 cars, 7.8ms
Speed: 1.7ms preprocess, 7.8ms inference, 1.5ms postprocess per image at shape (1, 3, 288, 640)


  0%|          | 1/1174 [00:00<02:53,  6.75it/s]


0: 288x640 10 cars, 6.5ms
Speed: 2.1ms preprocess, 6.5ms inference, 1.4ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 8 cars, 6.8ms
Speed: 2.4ms preprocess, 6.8ms inference, 1.6ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 7 cars, 8.7ms
Speed: 1.7ms preprocess, 8.7ms inference, 1.5ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 9 cars, 9.0ms
Speed: 2.8ms preprocess, 9.0ms inference, 1.8ms postprocess per image at shape (1, 3, 288, 640)


  0%|          | 5/1174 [00:00<00:55, 20.98it/s]

...


0: 288x640 1 person, 9 cars, 1 truck, 10.6ms
Speed: 2.6ms preprocess, 10.6ms inference, 1.8ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 1 person, 8 cars, 1 truck, 8.9ms
Speed: 2.6ms preprocess, 8.9ms inference, 1.4ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 1 person, 9 cars, 1 truck, 1 parking meter, 9.1ms
Speed: 2.5ms preprocess, 9.1ms inference, 1.4ms postprocess per image at shape (1, 3, 288, 640)


100%|█████████▉| 1172/1174 [00:42<00:00, 27.20it/s]


0: 288x640 9 cars, 1 truck, 10.2ms
Speed: 2.7ms preprocess, 10.2ms inference, 1.8ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 1 person, 7 cars, 1 truck, 9.7ms
Speed: 2.6ms preprocess, 9.7ms inference, 1.6ms postprocess per image at shape (1, 3, 288, 640)


100%|██████████| 1174/1174 [00:42<00:00, 27.41it/s]

ensemble.mp4 저장 완료


In [40]:
print("PeopleNet:", peoplenet_results)
print("TrafficNet:", trafficnet_results)
print("YOLO:", yolo_preds)


PeopleNet: [('person', [123, 50, 180, 210], 0.98), ('person', [210, 100, 265, 250], 0.91)]
TrafficNet: [('car', [310, 190, 420, 295], 0.93), ('truck', [30, 220, 80, 285], 0.77), ('traffic light', [450, 80, 470, 120], 0.81)]
YOLO: [('car', [0.3496551513671875, 429.36041259765625, 394.11248779296875, 718.3341064453125], 0.9042884111404419), ('car', [992.2640991210938, 403.11590576171875, 1441.1175537109375, 718.4762573242188], 0.8958342671394348), ('car', [583.782958984375, 432.5837707519531, 850.5471801757812, 648.3934326171875], 0.8771224617958069), ('car', [862.940673828125, 410.22283935546875, 1055.72607421875, 572.1493530273438], 0.8110697269439697), ('car', [545.53564453125, 434.90875244140625, 598.8132934570312, 516.465087890625], 0.5692992210388184), ('truck', [364.5845031738281, 383.34228515625, 532.461669921875, 590.4483032226562], 0.4804082214832306), ('car', [821.474609375, 436.8961181640625, 896.5811157226562, 524.8224487304688], 0.4115179777145386), ('car', [512.328125, 452

In [41]:
for box, score, lab in zip(boxes, fused_scores, fused_labels):
    lab_int = int(round(lab))
    class_name = id_to_class.get(lab_int, f"class{lab_int}")
    print(f"frame {frame_idx}: class={class_name}, box={box}, score={score}")  # print 추가!
    ...


frame 1174: class=person, box=[   0.076875    0.069444      0.1125     0.29167], score=0.32666666666666666
frame 1174: class=car, box=[    0.19375     0.26389      0.2625     0.40972], score=0.31
frame 1174: class=person, box=[    0.13125     0.13889     0.16562     0.34722], score=0.30333333333333334
frame 1174: class=car, box=[ 0.00021853     0.59633     0.24632     0.99769], score=0.3014294703801473
frame 1174: class=car, box=[    0.62017     0.55988      0.9007     0.99788], score=0.2986114223798116
frame 1174: class=car, box=[    0.36486     0.60081     0.53159     0.90055], score=0.2923741539319356
frame 1174: class=car, box=[    0.53934     0.56975     0.65983     0.79465], score=0.2703565756479899
frame 1174: class=traffic light, box=[    0.28125     0.11111     0.29375     0.16667], score=0.27
frame 1174: class=truck, box=[    0.01875     0.30556        0.05     0.39583], score=0.25666666666666665
frame 1174: class=car, box=[    0.34096     0.60404     0.37426     0.71731], sc

In [42]:
print(yolo_model.names)  # 'person', 'car', 'bus', 'truck', 'bicycle', 'traffic light', ...


{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68: 'microw

In [43]:
print("YOLO class_names:", class_names)
print("PeopleNet TrafficNet labels:", [c for c,_,_ in peoplenet_results + trafficnet_results])


YOLO class_names: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
PeopleNet TrafficNet labels: ['person', 'person', 'car', 'truck', 'traffic light']
